# Практика · Тема 18 · LSTM і GRU

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

> ⏱ Зошит навчає **тридцять три** маленькі мережі: 24 на синтетичній задачі
> й 9 мовних моделей на справжньому корпусі. Заміряно: **183 секунди
> процесорного часу** на чотирьох ядрах без відеокарти, з одним фіксованим
> потоком. На завантаженій машині «годинник на стіні» покаже втричі більше —
> це нормально, дивись на процесорний час, який друкує остання клітинка.

Що ми зробимо:

1. порахуємо **руками**, що робить із сигналом множник, менший за одиницю;
2. заміряємо, скільки градієнта доходить до **першого** кроку в RNN, LSTM і GRU;
3. напишемо клітинку LSTM самотужки й звіримо її з `nn.LSTM` до сьомого знака;
4. навчимо мережі на **синтетичній** задачі, де довжину залежності задаємо ми;
5. подивимось, як змінюються гейти й градієнт **після** навчання;
6. і чесно перевіримо на **нашому корпусі**, чи окупаються гейти на мовній моделі.

In [ ]:
# Фіксуємо потоки ДО імпорту numpy і torch: без цього процесорний час бреше
# в десятки разів — потоки OpenMP крутяться в очікуванні, і це рахується як робота.
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys
import time
import math
import glob
import gettext
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)

# з цієї миті рахуємо процесорний час усього зошита
started_at = time.process_time()

print('python', sys.version.split()[0])
print('torch ', torch.__version__)
print('numpy ', np.__version__)
print('потоків у torch:', torch.get_num_threads())

## 1. Множник, менший за одиницю

Уся тема тримається на одному шкільному факті. Якщо число, менше за одиницю,
множити саме на себе багато разів, воно летить до нуля — і летить **дуже**
швидко. Порахуймо це руками, без жодної мережі.

`decay` — це частка сигналу, яку крок пропускає далі. У звичайній рекурентній
мережі вона визначається вагами й не керована. У LSTM її задає **гейт забування**,
і мережа може її вивчити.

In [ ]:
# Скільки лишиться від сигналу після T кроків, якщо кожен крок множить його на decay.
steps_to_show = [5, 10, 25, 50, 100]

print('множник │', '  '.join('T=%-9d' % t for t in steps_to_show))
print('────────┼' + '─' * 62)
for decay in [0.5, 0.7, 0.9, 0.99, 1.0]:
    row = '  '.join('%-11.3e' % (decay ** t) for t in steps_to_show)
    print('  %.2f  │ %s' % (decay, row))

print()
# «Час напівжиття» навпаки: за скільки кроків сигнал слабшає у сто разів.
for decay in [0.5, 0.7, 0.9, 0.99]:
    steps = math.log(0.01) / math.log(decay)
    print('множник %.2f -> сигнал слабшає у 100 разів за %.1f кроків' % (decay, steps))

## 2. Скільки градієнта доходить до першого слова

Тепер замір на справжніх мережах. Ідея проста: подаємо на вхід випадкову
послідовність довжини `T`, беремо величину, порахувану на **останньому** кроці,
і питаємо в `autograd`, наскільки вона залежить від входу на **кожному** кроці.

Норма цієї похідної і є «скільки сигналу помилки доходить». Нас цікавить
відношення «на останньому кроці» до «на першому»: у скільки разів перший крок
чутніший за останній.

Мережі беремо **при ініціалізації**, ще не навчені: це замір самої конструкції,
а не того, чого вона навчилась.

In [ ]:
HIDDEN_GRAD = 64      # розмір прихованого стану
INPUT_GRAD = 32       # розмір вектора входу
CELL_CLASSES = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}


def fresh_cell(cell_name, seed, input_size=INPUT_GRAD, hidden_size=HIDDEN_GRAD):
    """Свіжа, ще не навчена мережа з відтворюваною ініціалізацією."""
    torch.manual_seed(seed)
    return CELL_CLASSES[cell_name](input_size, hidden_size, batch_first=True)


def gradient_per_step(cell, length, seed, input_size=INPUT_GRAD):
    """Норма похідної виходу останнього кроку по входу кожного кроку."""
    noise = torch.Generator().manual_seed(500 + seed)
    inputs = torch.randn(1, length, input_size, generator=noise, requires_grad=True)
    hidden_all, _ = cell(inputs)
    # беремо квадратичну величину на останньому кроці — аби було що диференціювати
    hidden_all[0, -1].pow(2).sum().backward()
    return inputs.grad[0].norm(dim=1).numpy().astype(float)


LENGTHS = [10, 25, 50, 100]
SEEDS = [0, 1, 2]
init_profiles = {}     # медіанний профіль по зернах — знадобиться далі
init_ratios = {}

print('у скільки разів останній крок чутніший за перший (три зерна)')
print('T    мережа  розкид по зернах')
for length in LENGTHS:
    for cell_name in CELL_CLASSES:
        profiles, ratios, zero_seeds = [], [], 0
        for seed in SEEDS:
            profile = gradient_per_step(fresh_cell(cell_name, seed), length, seed)
            profiles.append(profile)
            if profile[0] == 0.0:
                zero_seeds += 1          # градієнт до першого кроку занулився у float32
            else:
                ratios.append(profile[-1] / profile[0])
        key = (cell_name, length)
        init_profiles[key] = np.median(np.array(profiles), axis=0)
        init_ratios[key] = (sorted(ratios), zero_seeds)
        note = ' · НУЛЬ на %d зернах з 3' % zero_seeds if zero_seeds else ''
        print('%-4d %-7s %s%s' % (length, cell_name,
                                  ' '.join('%.3g' % v for v in sorted(ratios)) or '—',
                                  note))

Прочитаймо таблицю вголос, бо це головне число теми.

На десяти кроках звичайна RNN уже втрачає близько трьох порядків, а гейтовані
мережі — близько двох. На ста кроках у RNN **не лишається нічого**: похідна
по першому кроку дорівнює не «малому числу», а точному нулю — числа скінчилися
в межах `float32`. Перше слово не може вплинути на навчання **ніяк**.

Але друге, що видно в тій самій таблиці, підручники згадують рідше: LSTM і GRU
затухання **не скасовують**. Вони його вповільнюють — приблизно в мільйон разів
на 50 кроках, — і цього вистачає, щоб навчання зрушило. Порівняння в назві
«довга памʼять» — це порівняння з RNN, а не з нескінченністю.

In [ ]:
# Наочно: скільки порядків втрачається на кожні десять кроків.
print('втрата на 10 кроків, у порядках (log10):')
for cell_name in CELL_CLASSES:
    line = []
    for length in LENGTHS:
        ratios, zeros = init_ratios[(cell_name, length)]
        if not ratios:
            line.append('T=%-3d занулилось' % length)
            continue
        median_ratio = sorted(ratios)[len(ratios) // 2]
        per_ten = math.log10(median_ratio) / length * 10
        line.append('T=%-3d %5.2f' % (length, per_ten))
    print('%-5s' % cell_name, ' │ '.join(line))

## 3. Клітинка LSTM своїми руками

Формула LSTM виглядає страшно, поки її не набрати. Насправді це пʼять рядків:
три сигмоїди-гейти, один кандидат на запис і два оновлення стану.

Найкращий доказ, що всередині бібліотеки немає магії, — написати ці пʼять
рядків самотужки й звірити з `nn.LSTM`. `torch` складає всі чотири ворітні
вектори в один великий добуток, тому їх треба розрізати на чотири рівні шматки
в порядку **i, f, g, o** — вхідний гейт, гейт забування, кандидат, вихідний гейт.

In [ ]:
torch.manual_seed(7)
reference_lstm = nn.LSTM(6, 5, batch_first=True)   # 6 на вході, 5 у стані
sample = torch.randn(3, 9, 6)                      # 3 послідовності по 9 кроків

with torch.no_grad():
    reference_output, _ = reference_lstm(sample)

    weight_input = reference_lstm.weight_ih_l0
    weight_hidden = reference_lstm.weight_hh_l0
    bias_input = reference_lstm.bias_ih_l0
    bias_hidden = reference_lstm.bias_hh_l0

    hidden = torch.zeros(3, 5)     # h — те, що мережа віддає назовні
    memory = torch.zeros(3, 5)     # c — окрема стрічка памʼяті
    our_output = []
    for t in range(9):
        gates = sample[:, t] @ weight_input.T + bias_input \
              + hidden @ weight_hidden.T + bias_hidden
        input_gate, forget_gate, candidate, output_gate = gates.chunk(4, dim=1)
        # гейт забування вирішує, яку частку старої памʼяті лишити
        memory = torch.sigmoid(forget_gate) * memory \
               + torch.sigmoid(input_gate) * torch.tanh(candidate)
        hidden = torch.sigmoid(output_gate) * torch.tanh(memory)
        our_output.append(hidden)
    our_output = torch.stack(our_output, dim=1)

max_difference = float((our_output - reference_output).abs().max())
print('максимальне розходження з nn.LSTM:', max_difference)
assert torch.allclose(our_output, reference_output, atol=1e-6), 'розрахунок розійшовся!'
print('✅ збігається')

## 4. Задача, у якій довжину залежності задаємо ми

У справжньому тексті довга залежність трапляється рідко й нерівномірно. Щоб
побачити гейти в роботі, потрібна задача, де довжину можна **покрутити ручкою**.

Беремо найпростішу таку задачу. Послідовність довжини `T`: **перший** символ —
один із чотирьох «корисних», решта — випадкове сміття з інших восьми символів.
Мережа має на останньому кроці назвати перший символ. Тобто донести одне число
через `T − 1` крок, не загубивши його.

Монетка тут дає 0.25 — чотири рівноймовірні відповіді.

⚠️ Це **синтетика**. Вона показує механізм у чистому вигляді й нічого не каже
про наш корпус: справжню перевірку на корпусі ми зробимо в розділі 6.

In [ ]:
FIRST_SYMBOLS = 4     # скільки різних «корисних» перших символів
NOISE_SYMBOLS = 8     # скільки різних символів-наповнювачів
MEMO_VOCAB = FIRST_SYMBOLS + NOISE_SYMBOLS
MEMO_HIDDEN = 32
MEMO_EMBED = 16


def memo_batch(rng, size, length):
    """Партія прикладів: перший символ корисний, решта — сміття."""
    first = rng.integers(0, FIRST_SYMBOLS, size=size)
    sequence = rng.integers(FIRST_SYMBOLS, MEMO_VOCAB, size=(size, length))
    sequence[:, 0] = first
    return (torch.tensor(sequence, dtype=torch.long),
            torch.tensor(first, dtype=torch.long))


class MemoryNet(nn.Module):
    """Мережа для задачі на памʼять: ембединг → рекурентна клітинка → відповідь."""

    def __init__(self, cell_name, forget_bias=None):
        super().__init__()
        self.embedding = nn.Embedding(MEMO_VOCAB, MEMO_EMBED)
        self.cell = CELL_CLASSES[cell_name](MEMO_EMBED, MEMO_HIDDEN, batch_first=True)
        if forget_bias is not None:
            # другий чверток зсуву — це гейт забування; піднімаємо його,
            # щоб при народженні мережа памʼятала, а не забувала
            with torch.no_grad():
                self.cell.bias_ih_l0[MEMO_HIDDEN:2 * MEMO_HIDDEN].fill_(forget_bias)
                self.cell.bias_hh_l0[MEMO_HIDDEN:2 * MEMO_HIDDEN].fill_(0.0)
        self.head = nn.Linear(MEMO_HIDDEN, FIRST_SYMBOLS)

    def forward(self, sequence):
        hidden_all, _ = self.cell(self.embedding(sequence))
        return self.head(hidden_all[:, -1])       # відповідь читаємо з останнього кроку


def train_memory_net(cell_name, length, seed, forget_bias=None,
                     steps=600, batch_size=32, learning_rate=0.01):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = MemoryNet(cell_name, forget_bias)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    for _ in range(steps):
        sequence, target = memo_batch(rng, batch_size, length)
        optimizer.zero_grad()
        criterion(model(sequence), target).backward()
        optimizer.step()
    # перевіряємо на свіжих даних, яких мережа не бачила
    sequence, target = memo_batch(np.random.default_rng(1000 + seed), 2000, length)
    with torch.no_grad():
        accuracy = float((model(sequence).argmax(dim=1) == target).float().mean())
    return accuracy, model


print('готово: задача, мережа й навчання описані')

In [ ]:
# Чотири варіанти: звичайна RNN, LSTM як є, LSTM із піднятим гейтом забування, GRU.
MEMO_VARIANTS = [('RNN', 'RNN', None),
                 ('LSTM', 'LSTM', None),
                 ('LSTM +1', 'LSTM', 1.0),
                 ('GRU', 'GRU', None)]

memo_accuracy = {}
trained_memo = {}      # навчені мережі на T=25 знадобляться в наступному розділі

print('частка правильних відповідей, монетка = 0.2500')
print('T    варіант   зерно 0  зерно 1  зерно 2   медіана')
for length in [10, 25]:
    for label, cell_name, forget_bias in MEMO_VARIANTS:
        scores = []
        for seed in SEEDS:
            accuracy, model = train_memory_net(cell_name, length, seed, forget_bias)
            scores.append(accuracy)
            if length == 25:
                trained_memo[(label, seed)] = model
        memo_accuracy[(label, length)] = scores
        print('%-4d %-9s %s   %.4f' % (length, label,
                                       '  '.join('%.4f' % s for s in scores),
                                       sorted(scores)[1]))

Тут варто зупинитись, бо результат неприємний і саме тому цінний.

На десяти кроках задачу розвʼязують **усі**, включно зі звичайною RNN. На
двадцяти пʼяти картина розʼїжджається — і не так, як обіцяє підручник.
Дивись на числа вище: GRU тримає задачу на всіх трьох зернах, звичайна RNN
тримає її на двох із трьох, а **LSTM зі стандартною ініціалізацією не тримає
взагалі**, лишаючись на рівні монетки.

Причина не містична, і наступний розділ її вимірює: у щойно створеного LSTM
гейт забування стоїть посередині, тобто памʼять множиться приблизно на 0.5
щокроку. Один рядок, який піднімає зсув цього гейта на одиницю (варіант
«LSTM +1»), помітно міняє справу — при тому, що сама модель та сама.

## 5. Що змінюється після навчання

Замір із розділу 2 зроблено при ініціалізації. Але гейти на те й гейти, що
їхні значення **вивчаються**. Подивімось на ті самі мережі двічі: щойно
створену й ту саму після 600 кроків навчання на задачі з `T = 25`.

Міряємо дві речі: середнє значення гейта забування (у GRU його роль грає гейт
оновлення `z`) і те саме відношення градієнтів, що й у розділі 2.

In [ ]:
def memory_gate_stats(model, length=25, seed=0):
    """Гейт, що вирішує, скільки старої памʼяті лишити: середнє й частка відкритих."""
    rng = np.random.default_rng(2000 + seed)
    sequence, _ = memo_batch(rng, 256, length)
    embedded = model.embedding(sequence)
    cell = model.cell
    weight_input, weight_hidden = cell.weight_ih_l0, cell.weight_hh_l0
    bias_input, bias_hidden = cell.bias_ih_l0, cell.bias_hh_l0
    size = MEMO_HIDDEN
    hidden = torch.zeros(sequence.shape[0], size)
    memory = torch.zeros(sequence.shape[0], size)
    values = []
    with torch.no_grad():
        for t in range(length):
            from_input = embedded[:, t] @ weight_input.T + bias_input
            from_hidden = hidden @ weight_hidden.T + bias_hidden
            if isinstance(cell, nn.LSTM):
                gates = from_input + from_hidden
                input_gate, forget_gate, candidate, output_gate = gates.chunk(4, dim=1)
                forget_gate = torch.sigmoid(forget_gate)
                memory = forget_gate * memory \
                       + torch.sigmoid(input_gate) * torch.tanh(candidate)
                hidden = torch.sigmoid(output_gate) * torch.tanh(memory)
                values.append(forget_gate.flatten())
            else:                                    # GRU
                reset_in, update_in, new_in = from_input.chunk(3, dim=1)
                reset_hid, update_hid, new_hid = from_hidden.chunk(3, dim=1)
                reset_gate = torch.sigmoid(reset_in + reset_hid)
                update_gate = torch.sigmoid(update_in + update_hid)
                candidate = torch.tanh(new_in + reset_gate * new_hid)
                hidden = (1 - update_gate) * candidate + update_gate * hidden
                values.append(update_gate.flatten())
    all_values = torch.cat(values).numpy()
    # частка вентилів, відкритих майже навстіж: саме вони і є каналом памʼяті
    return float(all_values.mean()), float((all_values > 0.9).mean())


gate_before_after = {}
print('гейт памʼяті на вході задачі (T=25), три зерна')
print('варіант     середнє до      середнє після    відкритих >0.9 до   після')
for label, cell_name, forget_bias in MEMO_VARIANTS:
    if cell_name == 'RNN':
        continue                                    # у звичайної RNN гейтів немає
    mean_before, mean_after, open_before, open_after = [], [], [], []
    for seed in SEEDS:
        torch.manual_seed(seed)
        m0, o0 = memory_gate_stats(MemoryNet(cell_name, forget_bias))
        m1, o1 = memory_gate_stats(trained_memo[(label, seed)])
        mean_before.append(m0); open_before.append(o0)
        mean_after.append(m1); open_after.append(o1)
    gate_before_after[label] = (mean_before, mean_after, open_before, open_after)
    print('%-11s %.4f…%.4f   %.4f…%.4f    %.4f…%.4f   %.4f…%.4f' %
          (label, min(mean_before), max(mean_before),
           min(mean_after), max(mean_after),
           min(open_before), max(open_before),
           min(open_after), max(open_after)))

In [ ]:
def memo_gradient_per_step(model, length=25, seed=0):
    """Те саме, що в розділі 2, але на навченій мережі й на справжній втраті."""
    rng = np.random.default_rng(3000 + seed)
    sequence, target = memo_batch(rng, 64, length)
    embedded = model.embedding(sequence).detach().clone().requires_grad_(True)
    hidden_all, _ = model.cell(embedded)
    nn.CrossEntropyLoss()(model.head(hidden_all[:, -1]), target).backward()
    return embedded.grad.norm(dim=2).mean(dim=0).numpy().astype(float)


grad_before_after = {}
print('у скільки разів останній крок чутніший за перший, T=25 · три зерна')
print('варіант     до навчання                        після навчання')
for label, cell_name, forget_bias in MEMO_VARIANTS:
    profiles_before, profiles_after, ratio_before, ratio_after = [], [], [], []
    for seed in SEEDS:
        torch.manual_seed(seed)
        fresh_model = MemoryNet(cell_name, forget_bias)
        profile_0 = memo_gradient_per_step(fresh_model, 25, seed)
        profile_1 = memo_gradient_per_step(trained_memo[(label, seed)], 25, seed)
        profiles_before.append(profile_0)
        profiles_after.append(profile_1)
        ratio_before.append(profile_0[-1] / profile_0[0])
        ratio_after.append(profile_1[-1] / profile_1[0])
    grad_before_after[label] = (np.median(np.array(profiles_before), axis=0),
                                np.median(np.array(profiles_after), axis=0),
                                sorted(ratio_before), sorted(ratio_after))
    print('%-11s %-34s %s' %
          (label,
           ' '.join('%.4g' % v for v in sorted(ratio_before)),
           ' '.join('%.4g' % v for v in sorted(ratio_after))))

Ось те, чого не видно в підручнику: **гейти — не подарунок, а можливість**.
При народженні LSTM затухає не краще, ніж усе інше, і його гейт забування
стоїть коло 0.5. Після навчання на задачі, де памʼятати **треба**, гейт
підіймається, а відношення градієнтів падає на кілька порядків — тобто перший
крок стає чутнішим у стільки ж разів.

Тобто «LSTM лікує зникомий градієнт» — неточно. Точніше так: **LSTM дає мережі
ручку, якою вона може відкрити канал памʼяті, якщо задача цього вимагає.**
Якщо не вимагає — ручка лишається посередині.

## 6. А тепер чесно: наш корпус

Синтетика показала механізм. Але курс має відповісти на інше питання: чи
окупаються гейти **на нашому тексті**.

Корпус той самий, що й у всьому блоці, — українські переклади інтерфейсів
із системних файлів `.mo`. Спершу подивимось на його форму, бо саме вона
вирішує долю гейтів.

In [ ]:
# Канонічний токенізатор блоку: апостроф — звʼязка всередині слова, а не літера.
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
token_re = re.compile(TOKEN_PATTERN)


def load_corpus():
    """Українські переклади інтерфейсів, які вже лежать у системі."""
    texts = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as handle:
                catalog = gettext.GNUTranslations(handle)
            for source, target in catalog._catalog.items():
                if isinstance(source, str) and isinstance(target, str) \
                   and len(target) > 30 and 'Project-Id' not in target:
                    texts.append(target)
        except Exception:
            pass
    return texts


documents = load_corpus()
sentences = [token_re.findall(text.lower()) for text in documents]
lengths = np.array([len(s) for s in sentences])

corpus_is_usable = len(documents) >= 20000
print('документів:', len(documents))
print('слововживань:', int(lengths.sum()))
print('словоформ:', len({word for s in sentences for word in s}))
print('корпус придатний для навчання:', corpus_is_usable)
if not corpus_is_usable:
    print('⚠️ українська локаль на цій машині майже порожня.')
    print('   Розділи 6-7 нижче пропустяться, а синтетична частина вже відпрацювала.')

In [ ]:
if corpus_is_usable:
    print('медіана довжини речення:', int(np.median(lengths)), 'слів')
    print('90-й перцентиль:', int(np.percentile(lengths, 90)), 'слів')
    print('найдовше речення:', int(lengths.max()), 'слів')
    print()
    for limit in [10, 25, 50]:
        share = float((lengths > limit).mean())
        print('речень, довших за %2d слів: %6d = %.4f %%'
              % (limit, int((lengths > limit).sum()), 100 * share))
    length_histogram = [int((lengths == k).sum()) for k in range(0, 31)]
    print()
    print('скільки речень має рівно k слів, k = 1…15:')
    print(length_histogram[1:16])
else:
    print('пропущено: корпусу немає')

Ось де ховається відповідь. Медіана довжини — **шість слів**. Довгої залежності,
заради якої вигадали гейти, у цьому корпусі майже немає: щоб дотягтись до
першого слова, RNN треба пройти пʼять кроків, а не пʼятдесят.

Це не привід не міряти. Це привід поставити замір так, щоб він відповів
на питання прямо.

In [ ]:
if corpus_is_usable:
    kept = [s for s in sentences if 2 <= len(s) <= 30]
    from sklearn.model_selection import train_test_split
    train_sentences, valid_sentences = train_test_split(kept, test_size=0.1,
                                                        random_state=0)
    print('речень після відсіву 2…30 слів:', len(kept))
    print('навчальних:', len(train_sentences), '· перевірних:', len(valid_sentences))
else:
    train_sentences, valid_sentences = [], []
    print('пропущено: корпусу немає')

## 7. Мовна модель: три архітектури, три зерна, три епохи

Задача — та сама, що в темі 16: за початком речення вгадати наступне слово.
Мірка — **перплексія**: наскільки модель «здивована» правильним словом.
Менше означає краще.

Щоб три архітектури × три зерна × три епохи влізли в бюджет зошита, беремо
**12 000** навчальних речень із 82 тисяч. Це урізаний корпус, і абсолютні
числа тут менші, ніж у теми 16 на повному, — порівнювати можна лише
всередині цієї таблиці.

Головне, що ми **не** ріжемо: кількість зерен. Різниця, менша за розкид
по зернах, різницею не є.

In [ ]:
TRAIN_SENTENCES = 12000
VALID_SENTENCES = 3000
EMBED_SIZE = 64
HIDDEN_SIZE = 128
EPOCHS = 3
BATCH_SIZE = 64

if corpus_is_usable:
    train_part = train_sentences[:TRAIN_SENTENCES]
    valid_part = valid_sentences[:VALID_SENTENCES]

    word_counts = Counter(word for s in train_part for word in s)
    index_to_word = ['<pad>', '<eos>', '<unk>'] + sorted(
        word for word, count in word_counts.items() if count >= 5)
    word_to_index = {word: i for i, word in enumerate(index_to_word)}
    VOCAB_SIZE = len(index_to_word)

    print('навчальних речень:', len(train_part))
    print('перевірних речень:', len(valid_part))
    print('словник (min_count = 5):', VOCAB_SIZE)
else:
    VOCAB_SIZE = 0
    print('пропущено: корпусу немає')

In [ ]:
def encode(sentence_list):
    """Слова → номери, у кінці кожного речення — <eos>."""
    return [[word_to_index.get(word, 2) for word in s] + [1] for s in sentence_list]


def make_batches(encoded, batch_size):
    """Батчі з речень схожої довжини: інакше половина роботи йде на заповнювач."""
    order = sorted(range(len(encoded)), key=lambda i: len(encoded[i]))
    batches = []
    for start in range(0, len(order), batch_size):
        chunk = [encoded[i] for i in order[start:start + batch_size]]
        width = max(len(row) for row in chunk)
        block = torch.zeros(len(chunk), width, dtype=torch.long)
        for k, row in enumerate(chunk):
            block[k, :len(row)] = torch.tensor(row)
        batches.append(block)
    return batches


if corpus_is_usable:
    train_batches = make_batches(encode(train_part), BATCH_SIZE)
    valid_batches = make_batches(encode(valid_part), 128)
    valid_token_count = int(sum(int((b[:, 1:] != 0).sum()) for b in valid_batches))
    print('навчальних батчів:', len(train_batches))
    print('перевірних передбачень:', valid_token_count)
else:
    train_batches, valid_batches = [], []
    print('пропущено: корпусу немає')

In [ ]:
class WordLanguageModel(nn.Module):
    """Ембединг → рекурентна клітинка → розподіл на весь словник."""

    def __init__(self, cell_name):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_SIZE, padding_idx=0)
        self.cell = CELL_CLASSES[cell_name](EMBED_SIZE, HIDDEN_SIZE, batch_first=True)
        self.head = nn.Linear(HIDDEN_SIZE, VOCAB_SIZE)

    def forward(self, batch):
        hidden_all, _ = self.cell(self.embedding(batch))
        return self.head(hidden_all)


def perplexity(model):
    """Експонента середньої втрати на слово: у скільки разів модель здивована."""
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=0, reduction='sum')
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for block in valid_batches:
            inputs, targets = block[:, :-1], block[:, 1:]
            logits = model(inputs)
            total_loss += float(criterion(logits.reshape(-1, VOCAB_SIZE),
                                          targets.reshape(-1)))
            total_tokens += int((targets != 0).sum())
    return math.exp(total_loss / total_tokens)


def train_language_model(cell_name, seed):
    torch.manual_seed(seed)
    model = WordLanguageModel(cell_name)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    rng = np.random.default_rng(seed)
    curve = []
    for epoch in range(EPOCHS):
        model.train()
        for i in rng.permutation(len(train_batches)):
            block = train_batches[i]
            inputs, targets = block[:, :-1], block[:, 1:]
            optimizer.zero_grad()
            criterion(model(inputs).reshape(-1, VOCAB_SIZE),
                      targets.reshape(-1)).backward()
            optimizer.step()
        curve.append(perplexity(model))       # знімок після кожної епохи
    return curve, sum(p.numel() for p in model.parameters())


print('готово: мовна модель і навчання описані')

In [ ]:
language_curves = {}
language_params = {}

if corpus_is_usable:
    print('перплексія на перевірній частині після кожної епохи')
    print('мережа зерно  епоха 1  епоха 2  епоха 3   ваг      час, с')
    for cell_name in CELL_CLASSES:
        curves = []
        for seed in SEEDS:
            started = time.process_time()
            curve, parameter_count = train_language_model(cell_name, seed)
            curves.append(curve)
            print('%-6s %d     %s   %-8d %.1f'
                  % (cell_name, seed, '  '.join('%7.2f' % v for v in curve),
                     parameter_count, time.process_time() - started))
        language_curves[cell_name] = curves
        language_params[cell_name] = parameter_count
else:
    print('пропущено: корпусу немає')

## 8. Купи по зернах, а не різниця середніх

Різниця двох середніх нічого не варта, поки не відомий розкид. Тому дивимось
на **купу** — від найгіршого зерна до найкращого. Якщо купи двох мереж
перетинаються, різниці немає, і так і треба сказати.

In [ ]:
if corpus_is_usable:
    for epoch in range(EPOCHS):
        print('── після епохи %d ──' % (epoch + 1))
        piles = {}
        for cell_name in CELL_CLASSES:
            values = sorted(curve[epoch] for curve in language_curves[cell_name])
            piles[cell_name] = values
            print('  %-5s медіана %7.2f   купа %7.2f…%-7.2f'
                  % (cell_name, values[1], values[0], values[-1]))
        names = list(CELL_CLASSES)
        for a in range(len(names)):
            for b in range(a + 1, len(names)):
                first, second = piles[names[a]], piles[names[b]]
                overlap = not (first[-1] < second[0] or second[-1] < first[0])
                print('  %-5s проти %-5s: %s' % (names[a], names[b],
                      'купи перетинаються — різниці немає' if overlap
                      else 'купи не перетинаються — різниця справжня'))
else:
    print('пропущено: корпусу немає')

In [ ]:
if corpus_is_usable:
    # Що важить більше: ще одна епоха чи вибір архітектури?
    best_by_epoch, worst_by_epoch = [], []
    for epoch in range(EPOCHS):
        medians = [sorted(c[epoch] for c in language_curves[name])[1]
                   for name in CELL_CLASSES]
        best_by_epoch.append(min(medians))
        worst_by_epoch.append(max(medians))

    epoch_gain = best_by_epoch[0] - best_by_epoch[1]
    architecture_gap = worst_by_epoch[-1] - best_by_epoch[-1]
    print('виграш від другої епохи (за найкращою архітектурою): %.2f пункта'
          % epoch_gain)
    print('розрив між найкращою й найгіршою архітектурою на 3 епохах: %.2f пункта'
          % architecture_gap)
    print('ще одна епоха важить у %.1f раза більше за вибір архітектури'
          % (epoch_gain / architecture_gap))
    print()
    for cell_name in CELL_CLASSES:
        print('%-5s ваг: %d' % (cell_name, language_params[cell_name]))
else:
    print('пропущено: корпусу немає')

## 9. Підсумок числами

Складімо все, що заміряли, в один короткий висновок — і назвімо конфігурацію,
бо без неї жодне з цих чисел не існує.

In [ ]:
print('КОНФІГУРАЦІЯ')
print('  градієнт: H = %d, вхід %d, три зерна, мережі при ініціалізації'
      % (HIDDEN_GRAD, INPUT_GRAD))
print('  синтетика: H = %d, 600 кроків, батч 32, Adam 0.01, три зерна'
      % MEMO_HIDDEN)
if corpus_is_usable:
    print('  мовна модель: E = %d, H = %d, батч %d, Adam 2e-3, %d епохи, %d речень'
          % (EMBED_SIZE, HIDDEN_SIZE, BATCH_SIZE, EPOCHS, TRAIN_SENTENCES))
print()
print('ЩО ЗАМІРЯНО')
ratios_50, zeros_50 = init_ratios[('RNN', 50)]
print('  на 50 кроках RNN слабша до першого кроку у %.3g раза' % sorted(ratios_50)[1])
ratios_50_lstm, _ = init_ratios[('LSTM', 50)]
print('  на 50 кроках LSTM — у %.3g раза' % sorted(ratios_50_lstm)[1])
_, zeros_100 = init_ratios[('RNN', 100)]
print('  на 100 кроках градієнт RNN до першого кроку — нуль на %d зернах з 3'
      % zeros_100)
print('  синтетика T=25: GRU медіана %.4f, RNN %.4f, LSTM %.4f'
      % (sorted(memo_accuracy[('GRU', 25)])[1],
         sorted(memo_accuracy[('RNN', 25)])[1],
         sorted(memo_accuracy[('LSTM', 25)])[1]))
if corpus_is_usable:
    for cell_name in CELL_CLASSES:
        values = sorted(curve[-1] for curve in language_curves[cell_name])
        print('  корпус, 3 епохи, %-5s медіана %.2f купа %.2f…%.2f'
              % (cell_name, values[1], values[0], values[-1]))
print()
print('процесорний час усього зошита: %.1f с' % (time.process_time() - started_at))

## Завдання

### 🟢 Рівень 1
Додай у замір градієнта довжину `T = 200` і подивись, чи лишається щось
у LSTM і GRU. **Зроблено, якщо** ти назвеш, на скількох зернах із трьох
кожна з мереж занулилась.

### 🟡 Рівень 2
Візьми варіант `LSTM +1` і покрути значення зсуву гейта забування: 0, 1, 2, 3.
Для кожного значення заміряй частку правильних на `T = 25` по трьох зернах.
**Зроблено, якщо** ти побудуєш таблицю «зсув → медіана й купа» і скажеш,
чи є значення, при якому LSTM обганяє GRU.

### 🔴 Рівень 3
Зроби мовну модель, у якій довга залежність **є**: залиш у корпусі лише
речення, довші за 15 слів, і повтори порівняння трьох архітектур на трьох
зернах. **Зроблено, якщо** ти скажеш, чи змінився порядок мереж порівняно
з розділом 8, і покажеш купи по зернах, а не самі медіани.